# Working with images using GPT-4 Vision model


## What You Will Learn

- **GPT-4 Vision model**: Discover the vision capabilities of GPT-4 model and how to build computer vision applications with it.


## Getting Started

Before we jump in, ensure you have:

- A Google Colab account.
- Basic knowledge of Python and REST APIs.
- An OpenAI API key with access to the DALL-E service ([OpenAI](https://platform.openai.com/account/api-keys)).




# 2. Libraries import

In [ ]:
!pip install openai

In [ ]:
# Colab: pull your key from the Secrets manager into the runtime
import os
try:
    from google.colab import userdata  # works in Colab
    key = userdata.get("OPENAI_API_KEY")  # stored under the 🔑 tab
except Exception:
    key = None

if not key:
    raise RuntimeError(
        "Missing OPENAI_API_KEY in Colab Secrets (click the 🔑 icon and add it)."
    )

# Basic sanity check: ensure this is a *project-scoped* key (recommended)
# New keys usually look like: sk-proj-************************
print("Key prefix:", (key[:12] + "…") if key else "None")
os.environ["OPENAI_API_KEY"] = key


Key prefix: sk-proj-s4Im…


In [ ]:
import os
import openai
import base64
import requests

from openai import OpenAI

# 3. Sending a first request to OpenAI API


### 3.1 Setting up API Key

In [ ]:
client = OpenAI()

# 4. Classifing and describing images



In [20]:
# 1. Download the image
image_http_url = "https://pngimg.com/uploads/rose/rose_PNG67024.png"
img_bytes = requests.get(image_http_url).content

# 2. Encode as base64 data URL
b64_image = base64.b64encode(img_bytes).decode("utf-8")
data_url = f"data:image/jpeg;base64,{b64_image}"

# 3. Call a vision-capable model
response = client.chat.completions.create(
    model="gpt-4.1-mini",   # or gpt-4o-mini / another vision model
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What's in this image?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": data_url
                    }
                },
            ],
        }
    ],
    max_tokens=100,

)

print(response.choices[0].message.content)

The image shows a bouquet of roses, consisting of red and pink roses. There are also some green fern leaves and other green foliage interspersed among the flowers. The background is transparent, highlighting the bouquet.


In [27]:
# 1. Download the image
#image_http_url = "https://www.citypng.com/public/uploads/preview/fresh-red-tomato-slice-hd-transparent-background-735811696678150cgokapqhm1.png"
image_http_url = "https://image.similarpng.com/file/similarpng/original-picture/2020/08/Tomato-sauce-bowl-with-fresh-tomato-on-transparent-background-PNG.png"

img_bytes = requests.get(image_http_url).content

# 2. Encode as base64 data URL
b64_image = base64.b64encode(img_bytes).decode("utf-8")
data_url = f"data:image/jpeg;base64,{b64_image}"


response = client.chat.completions.create(
    model="gpt-4.1-mini",   # or gpt-4o-mini / another vision model
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Act as a image classification algorithm. Your task is to classify this image inside one of these classes: Tomato, other. Provide only classes, and nothing else"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": data_url
                    }
                },
            ],
        }
    ],
    max_tokens=300,
)

print(response.choices[0].message.content)

Tomato


In [28]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="generate an image of a glass cabinet with the most popular semi-precious stones",
    tools=[{"type": "image_generation"}],
)

# Save the image to a file
image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    image_base64 = image_data[0]
    with open("luxuryCabinet.png", "wb") as f:
        f.write(base64.b64decode(image_base64))

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Your organization must be verified to use the model `gpt-4.1-mini`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': None, 'code': None}}

## Text To Speech using TTS API

In [31]:
speech_file_path = "speech.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Today is a wonderful day to build something people love!",
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file(speech_file_path)